# Importar librerías

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    from_json,
    when
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

from IPython.display import display

# Crear SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("MonitoreoSensores")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)

Spark iniciado correctamente
Versión de Spark: 3.4.0


# Definir el esquema de los sensores

In [3]:
schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("maquina_id", StringType(), True),
    StructField("timestamp", IntegerType(), True),
    StructField("temperatura", DoubleType(), True),
    StructField("humedad", DoubleType(), True),
    StructField("presion", DoubleType(), True)
])

print("Esquema de sensores definido correctamente")

Esquema de sensores definido correctamente


# Conectar con Kafka

In [5]:
df_kafka = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        "kafka:9092"
    )
    .option(
        "subscribe",
        "actividad-topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

print("Conexión con Kafka preparada")
print("Topic:", "actividad-topic")
print("Esperando eventos...")

Conexión con Kafka preparada
Topic: actividad-topic
Esperando eventos...


# Convertir el JSON recibido

In [6]:
df_sensores = (
    df_kafka
    .selectExpr("CAST(value AS STRING) AS json")
    .select(
        from_json(
            col("json"),
            schema
        ).alias("data")
    )
    .select("data.*")
)

print("Transformación JSON preparada")

Transformación JSON preparada


# Definir los umbrales

In [7]:
umbrales_data = [
    ("M001", 30.0, 70.0, 1000.0),
    ("M002", 35.0, 75.0, 1000.0),
    ("M003", 28.0, 65.0, 1005.0),
    ("M004", 32.0, 80.0, 995.0),
    ("M005", 30.0, 70.0, 1005.0)
]

df_umbrales = spark.createDataFrame(
    umbrales_data,
    [
        "maquina_id",
        "temp_max",
        "humedad_max",
        "presion_min"
    ]
)

print("Umbrales definidos:")
df_umbrales.show()

Umbrales definidos:
+----------+--------+-----------+-----------+
|maquina_id|temp_max|humedad_max|presion_min|
+----------+--------+-----------+-----------+
|      M001|    30.0|       70.0|     1000.0|
|      M002|    35.0|       75.0|     1000.0|
|      M003|    28.0|       65.0|     1005.0|
|      M004|    32.0|       80.0|      995.0|
|      M005|    30.0|       70.0|     1005.0|
+----------+--------+-----------+-----------+



# Cruzar sensores con los umbrales

In [8]:
df_comparacion = (
    df_sensores
    .join(
        df_umbrales,
        on="maquina_id",
        how="left"
    )
)

print("Cruce entre sensores y umbrales preparado")

Cruce entre sensores y umbrales preparado


# Calcular las alertas

In [9]:
df_alertas = (
    df_comparacion

    .withColumn(
        "alerta_temperatura",
        when(
            col("temperatura") > col("temp_max"),
            1
        ).otherwise(0)
    )

    .withColumn(
        "alerta_humedad",
        when(
            col("humedad") > col("humedad_max"),
            1
        ).otherwise(0)
    )

    .withColumn(
        "alerta_presion",
        when(
            col("presion") < col("presion_min"),
            1
        ).otherwise(0)
    )
)

df_alertas = df_alertas.withColumn(
    "total_alertas",
    col("alerta_temperatura")
    + col("alerta_humedad")
    + col("alerta_presion")
)

print("Reglas de alerta preparadas")

Reglas de alerta preparadas


# Determinar el estado de las alertas

In [11]:
df_resultado = (
    df_alertas

    .withColumn(
        "estado",
        when(
            col("total_alertas") == 0,
            "NORMAL"
        )
        .when(
            col("total_alertas") == 1,
            "ALERTA"
        )
        .otherwise(
            "CRITICO"
        )
    )

    .withColumn(
        "tipo_alerta",
        when(
            col("total_alertas") == 0,
            "NINGUNA"
        )
        .when(
            (col("alerta_temperatura") == 1) &
            (col("alerta_humedad") == 0) &
            (col("alerta_presion") == 0),
            "TEMPERATURA"
        )
        .when(
            (col("alerta_temperatura") == 0) &
            (col("alerta_humedad") == 1) &
            (col("alerta_presion") == 0),
            "HUMEDAD"
        )
        .when(
            (col("alerta_temperatura") == 0) &
            (col("alerta_humedad") == 0) &
            (col("alerta_presion") == 1),
            "PRESION"
        )
        .otherwise(
            "MULTIPLE"
        )
    )
)

print("Clasificación NORMAL / ALERTA / CRITICO preparada")

Clasificación NORMAL / ALERTA / CRITICO preparada


# Seleccionar el resultado final

In [12]:
df_resultado_final = df_resultado.select(
    "sensor_id",
    "maquina_id",
    "timestamp",
    "temperatura",
    "humedad",
    "presion",
    "temp_max",
    "humedad_max",
    "presion_min",
    "alerta_temperatura",
    "alerta_humedad",
    "alerta_presion",
    "total_alertas",
    "estado",
    "tipo_alerta"
)

print("DataFrame final preparado")

DataFrame final preparado


# Preparar las rutas de salida

In [13]:
output_path = "work/jupyter_storage/resultado_sensores"

checkpoint_console = "work/jupyter_storage/checkpoints/sensores_console"

checkpoint_parquet = "work/jupyter_storage/checkpoints/sensores_parquet"

print("Ruta Parquet:")
print(output_path)

Ruta Parquet:
work/jupyter_storage/resultado_sensores


# Iniciar el streaming. Esta es la celda que se debe ejecutar antes de iniciar el Producer.

In [14]:
print("=" * 70)
print("        CONSUMER DE SENSORES INICIADO")
print("=" * 70)

print()
print("Kafka está esperando eventos...")
print()
print(">>> AHORA INICIAR EL PRODUCER <<<")
print()
print("Ejecutar el Producer de sensores en la terminal.")
print("Déjar generar eventos durante unos segundos.")
print()
print("Los eventos aparecerán por pantalla automáticamente.")
print("También serán almacenados en formato Parquet.")
print()
print("=" * 70)

        CONSUMER DE SENSORES INICIADO

Kafka está esperando eventos...

>>> AHORA INICIAR EL PRODUCER <<<

Ejecutar el Producer de sensores en la terminal.
Déjar generar eventos durante unos segundos.

Los eventos aparecerán por pantalla automáticamente.
También serán almacenados en formato Parquet.



# Mostrar las notificaciones por pantalla - Este es el primer writeStream

In [15]:
query_console = (
    df_resultado_final
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 20)
    .option(
        "checkpointLocation",
        checkpoint_console
    )
    .trigger(processingTime="2 seconds")
    .start()
)

print("Stream de notificaciones iniciado.")
print("Esperando eventos del Producer...")

Stream de notificaciones iniciado.
Esperando eventos del Producer...


# Guardar los resultados en Parquet - Una vez iniciado el stream de consola, ejecutamos el segundo stream:

In [16]:
query_parquet = (
    df_resultado_final
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "path",
        output_path
    )
    .option(
        "checkpointLocation",
        checkpoint_parquet
    )
    .trigger(processingTime="2 seconds")
    .start()
)

print("Stream Parquet iniciado.")
print()
print("Los eventos recibidos se están almacenando en:")
print(output_path)

Stream Parquet iniciado.

Los eventos recibidos se están almacenando en:
work/jupyter_storage/resultado_sensores


# Esperar y verificar que los streams están activos

In [17]:
print("Streams activos:")

for q in spark.streams.active:
    print(
        "ID:", q.id,
        "| Estado:", q.status
    )

Streams activos:
ID: 0881fa14-8399-4cb9-ae4a-b680350f6736 | Estado: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
ID: d0ef5aad-dbf4-452f-81c6-f66624102ee6 | Estado: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


# Detener el consumer

In [18]:
query_console.stop()
query_parquet.stop()

print("=" * 70)
print("        STREAMING DETENIDO")
print("=" * 70)

print()
print("Los eventos recibidos han quedado almacenados en Parquet.")
print()
print("Ruta:")
print(output_path)

        STREAMING DETENIDO

Los eventos recibidos han quedado almacenados en Parquet.

Ruta:
work/jupyter_storage/resultado_sensores


# Leer el Parquet

In [19]:
df_parquet = spark.read.parquet(output_path)

print("Parquet leído correctamente.")
print()

print("Cantidad total de registros:")
print(df_parquet.count())

Parquet leído correctamente.

Cantidad total de registros:
22


# Mostrar las últimas 10 ejecuciones

In [20]:
df_ultimos_10 = (
    df_parquet
    .orderBy(col("timestamp").desc())
    .limit(10)
)

print("=" * 100)
print("                 RESULTADO FINAL - ÚLTIMAS 10 EJECUCIONES")
print("=" * 100)

df_ultimos_10.show(
    10,
    truncate=False
)

                 RESULTADO FINAL - ÚLTIMAS 10 EJECUCIONES
+---------+----------+----------+-----------+-------+-------+--------+-----------+-----------+------------------+--------------+--------------+-------------+-------+-----------+
|sensor_id|maquina_id|timestamp |temperatura|humedad|presion|temp_max|humedad_max|presion_min|alerta_temperatura|alerta_humedad|alerta_presion|total_alertas|estado |tipo_alerta|
+---------+----------+----------+-----------+-------+-------+--------+-----------+-----------+------------------+--------------+--------------+-------------+-------+-----------+
|S004     |M004      |1791072333|25.19      |74.37  |994.36 |32.0    |80.0       |995.0      |0                 |0             |1             |1            |ALERTA |PRESION    |
|S002     |M002      |1791072330|38.38      |85.77  |995.87 |35.0    |75.0       |1000.0     |1                 |1             |1             |3            |CRITICO|MULTIPLE   |
|S005     |M005      |1791072327|31.18      |84.13  

# Resumen final

In [21]:
total_registros = df_parquet.count()

print()
print("=" * 70)
print("                 RESULTADO FINAL DEL EJERCICIO")
print("=" * 70)

print()
print(f"Total de eventos procesados: {total_registros}")

print()
print("Eventos por estado:")

(
    df_parquet
    .groupBy("estado")
    .count()
    .orderBy("estado")
    .show()
)

print("Ubicación de los resultados Parquet:")
print(output_path)

print()
print("Últimas 10 ejecuciones:")
df_ultimos_10.show(10, truncate=False)

print("=" * 70)
print("              PROCESAMIENTO FINALIZADO")
print("=" * 70)


                 RESULTADO FINAL DEL EJERCICIO

Total de eventos procesados: 22

Eventos por estado:
+-------+-----+
| estado|count|
+-------+-----+
| ALERTA|    8|
|CRITICO|   10|
| NORMAL|    4|
+-------+-----+

Ubicación de los resultados Parquet:
work/jupyter_storage/resultado_sensores

Últimas 10 ejecuciones:
+---------+----------+----------+-----------+-------+-------+--------+-----------+-----------+------------------+--------------+--------------+-------------+-------+-----------+
|sensor_id|maquina_id|timestamp |temperatura|humedad|presion|temp_max|humedad_max|presion_min|alerta_temperatura|alerta_humedad|alerta_presion|total_alertas|estado |tipo_alerta|
+---------+----------+----------+-----------+-------+-------+--------+-----------+-----------+------------------+--------------+--------------+-------------+-------+-----------+
|S004     |M004      |1791072333|25.19      |74.37  |994.36 |32.0    |80.0       |995.0      |0                 |0             |1             |1     